Exercise 1: Indexing
Use Elasticsearch to index and query the grocery dataset (directory data/grocery). The dataset contains
a catalogue of items sold in a store. Indications on how to perform the indexing and querying are
provided next. Write your code in a single Python notebook. You are free to use the Elasticsearch
wrapper API, the custom helpers we have used in class, or both. For each of the points below, write
some brief comments (or text cells) in your notebook to describe how you addressed that specific point.
Use the numbering provided below to refer to the points in your notebook (e.g., “Point #1: I took care
of efficient loading by ...”)
Indexing. Please index the dataset taking into account the following points:
1. The dataset needs to be loaded efficiently.
2. This is intended to be a rather dynamic database, where updates are done multiple times per hour
3. Create an index with 2 replicas.
4. The documents should allow full text search on the item name and category name, and should allow
filtering and sorting by category and price.
5. A relevance metric that allows for text length discounting should be used. Increase by 30% the amount
of length discounting compared to the default value.


Querying. For each of the points below, write a separate Python parametric function that performs:


6. An exact-match query on the item name.
7. A full-text query on item name and category, with category boosted by a factor 4 compared to item
name.
8. A full-text query on the category that sorts the results by price (lowest to highest).
9. A fuzzy query on the item name.
Test your functions with at least one example query each.

In [68]:
import pandas as pd
from elasticsearch import Elasticsearch, helpers

In [69]:
# Since the docker image uses elasticsearch 8.15.3, my local env also installs a es version 8, 
# as it is not compatible with version 9

# Connect to local Elasticsearch instance
client = Elasticsearch(
  "https://localhost:9200",
  ca_certs="./http_ca.crt",
  basic_auth=("elastic", "mysecurepassword")
)

# Should provide a response with a cluster instance and name
client.info()


ObjectApiResponse({'name': '8601b1815693', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'yduMnDUwTwaSmPbO16TF6A', 'version': {'number': '8.15.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'f97532e680b555c3a05e73a74c28afb666923018', 'build_date': '2024-10-09T22:08:00.328917561Z', 'build_snapshot': False, 'lucene_version': '9.11.1', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [70]:
client.indices.delete(index="grocery-index")

ObjectApiResponse({'acknowledged': True})

In [71]:
# To load the dataset efficiently, the data should be loaded in bulk
# Therefore the index should have automatic refreshing disabled
# i.e set refresh_interval: -1 (remember to update it at after data has been inserted) 
client.indices.create(
  index="grocery-index",
  settings={
    "number_of_shards": 4,
    "number_of_replicas": 2,
    "refresh_interval": -1,
    "index": {
      "similarity": {
        "default": {
          "type": "BM25",
          "b": 0.975 # Default value in Elasticsearch is 0,75 https://www.elastic.co/docs/reference/elasticsearch/index-settings/similarity
        }
      }
    }
  },
  mappings={
    "properties": {
      "product_name": {
       "type": "text",
      },
      "category": {
        "type": "text",
        "fields": {
          "raw": {
            "type": "keyword"
          } 
        }
      },
      "price": {
        "type": "float"
      }
    }
  }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'grocery-index'})

In [72]:
client.indices.get(index="grocery-index")

ObjectApiResponse({'grocery-index': {'aliases': {}, 'mappings': {'properties': {'category': {'type': 'text', 'fields': {'raw': {'type': 'keyword'}}}, 'price': {'type': 'float'}, 'product_name': {'type': 'text'}}}, 'settings': {'index': {'routing': {'allocation': {'include': {'_tier_preference': 'data_content'}}}, 'refresh_interval': '-1', 'number_of_shards': '4', 'provided_name': 'grocery-index', 'similarity': {'default': {'type': 'BM25', 'b': '0.975'}}, 'creation_date': '1779636722441', 'number_of_replicas': '2', 'uuid': '-odQzTLnRUeN4aisiJlO4w', 'version': {'created': '8512000'}}}}})

In [73]:
# Create a helper function that reads the csv file and prepares it for a bulk insert into the index
def bulk_index(index_name="grocery-index"):
  df = pd.read_csv("../../data/grocery/products.csv")
  
  # Turn the csv file into a dictionary, so we can fetch the value for each row, by its column key
  # https://www.geeksforgeeks.org/python/pandas-dataframe-to_dict/
  df_dictionary = df.to_dict(orient="records")
  
  # Use python generator function to load the dataset efficiently
  for record in df_dictionary:
    yield {
      "_index": index_name,
      "_id": record['product_id'],
      "_source": {
        "product_name": record['product_name'],
        "aisle_id": record['aisle_id'],
        "department_id": record['department_id'],
        "category": record['category'],
        "price": float(record['price'])
      }
    }

# Code is inspired by: https://stackoverflow.com/questions/71889063/bulk-index-create-documents-with-elasticsearch-for-python 
# & https://www.geeksforgeeks.org/elasticsearch/using-the-elasticsearch-bulk-api-for-high-performance-indexing/
helpers.bulk(client, bulk_index())

(49688, [])

In [74]:
# Sanity check to see if I actually have all the parameters for each entry in the csv. 
client.get(index="grocery-index", id="10")

ObjectApiResponse({'_index': 'grocery-index', '_id': '10', '_version': 1, '_seq_no': 5, '_primary_term': 1, 'found': True, '_source': {'product_name': 'Sparkling Orange Juice & Prickly Pear Beverage', 'aisle_id': 115, 'department_id': 7, 'category': 'Beverages', 'price': 5.38}})

In [75]:
# Update refresh_interval back to default for ElasticSearch so it now is ready to handle updates done multiple times an hour
client.indices.put_settings(
  index="grocery-index",
  settings={
    "index": {
      "refresh_interval": "5s"
    }
  }
)

ObjectApiResponse({'acknowledged': True})

In [76]:
# Point 6: An exact-match query on the item name
client.search(
  index="grocery-index",
  query={
    "match_phrase": {
      "product_name": "Fresh Breath Oral Rinse Mild Mint"
    }
  }
)

ObjectApiResponse({'took': 19, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1, 'relation': 'eq'}, 'max_score': 31.73388, 'hits': [{'_index': 'grocery-index', '_id': '22', '_score': 31.73388, '_source': {'product_name': 'Fresh Breath Oral Rinse Mild Mint', 'aisle_id': 20, 'department_id': 11, 'category': 'Personal Care', 'price': 8.07}}]}})

In [81]:
# Point 7: A full-text query on item name and category, with category boosted by a factor 4 compared to item name
# https://www.elastic.co/docs/reference/query-languages/query-dsl/query-dsl-multi-match-query
client.search(
  index="grocery-index",
  query={
    "multi_match": {
      "query": "fruits and turkey burger",
      "fields": ["product_name", "category^4"]
    }
  }
)

ObjectApiResponse({'took': 6, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 6343, 'relation': 'eq'}, 'max_score': 15.746758, 'hits': [{'_index': 'grocery-index', '_id': '36157', '_score': 15.746758, '_source': {'product_name': 'Turkey Burger', 'aisle_id': 35, 'department_id': 12, 'category': 'Meat & Seafood', 'price': 12.76}}, {'_index': 'grocery-index', '_id': '36214', '_score': 13.5380745, '_source': {'product_name': 'Turkey Burger Patties', 'aisle_id': 49, 'department_id': 12, 'category': 'Meat & Seafood', 'price': 13.72}}, {'_index': 'grocery-index', '_id': '40781', '_score': 13.5380745, '_source': {'product_name': 'Seasoned Turkey Burger', 'aisle_id': 34, 'department_id': 1, 'category': 'Frozen Foods', 'price': 7.31}}, {'_index': 'grocery-index', '_id': '9', '_score': 11.898925, '_source': {'product_name': 'Light Strawberry Blueberry Yogurt', 'aisle_id': 120, 'department_id': 16, 'category': 'Fruits', 'price': 

In [78]:
# # Point 8: A full-text query on the category that sorts the results by price (lowest to highest).
client.search(
  index="grocery-index",
  query={
    "match": {
      "category": "Beverages"
    }
  },
  sort=[
    {
      "price": {
        "order": "asc"
      }
    }
  ]
)

ObjectApiResponse({'took': 12, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1733, 'relation': 'eq'}, 'max_score': None, 'hits': [{'_index': 'grocery-index', '_id': '3689', '_score': None, '_source': {'product_name': 'Beef Broth', 'aisle_id': 69, 'department_id': 15, 'category': 'Beverages', 'price': 1.99}, 'sort': [1.99]}, {'_index': 'grocery-index', '_id': '10805', '_score': None, '_source': {'product_name': 'Gold Nutrition Energy Bar Chocolate Peanut Butter', 'aisle_id': 3, 'department_id': 19, 'category': 'Beverages', 'price': 1.99}, 'sort': [1.99]}, {'_index': 'grocery-index', '_id': '22265', '_score': None, '_source': {'product_name': 'Lavender with Baking Soda & Alpine Lichen Deodorant Stick', 'aisle_id': 25, 'department_id': 11, 'category': 'Beverages', 'price': 1.99}, 'sort': [1.99]}, {'_index': 'grocery-index', '_id': '22133', '_score': None, '_source': {'product_name': 'Lindor Assorted Chocolate Truffles'

In [79]:
# Point 9: A fuzzy query on the item name.
client.search(
  index="grocery-index",
  query={
    "match": {
      "product_name": "Fresh Breath Oral Rinse Mild Mint"
    }
  }
)

ObjectApiResponse({'took': 14, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 715, 'relation': 'eq'}, 'max_score': 31.733877, 'hits': [{'_index': 'grocery-index', '_id': '22', '_score': 31.733877, '_source': {'product_name': 'Fresh Breath Oral Rinse Mild Mint', 'aisle_id': 20, 'department_id': 11, 'category': 'Personal Care', 'price': 8.07}}, {'_index': 'grocery-index', '_id': '27624', '_score': 20.88214, '_source': {'product_name': 'Icy Mint Oral Rinse', 'aisle_id': 20, 'department_id': 11, 'category': 'Personal Care', 'price': 3.89}}, {'_index': 'grocery-index', '_id': '39295', '_score': 15.604292, '_source': {'product_name': 'Fresh Breath Mint-Coated Capsules', 'aisle_id': 20, 'department_id': 11, 'category': 'Personal Care', 'price': 10.77}}, {'_index': 'grocery-index', '_id': '31709', '_score': 15.574554, '_source': {'product_name': 'Dry Mouth Oral Rinse', 'aisle_id': 20, 'department_id': 11, 'category': 'Person